# 第 4 週 實作｜分部積分

換元是鏈鎖法則反過來走,分部是<strong>乘積法則反過來走</strong>。有了這兩招,課本上大部分的積分都對付得了——但也會第一次遇到「真的算不出來」的積分。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜遞迴公式寫成遞迴函式

觀念 6 的遞迴公式 $I_n=x^n e^x-n I_{n-1}$ 有 base case、有遞迴關係——這就是程式課的遞迴。三行寫出來,再和 SymPy 對答案。


In [ ]:
x = sp.Symbol('x')

def I(n):
    """∫ x^n e^x dx —— 遞迴公式的直譯"""
    if n == 0:
        return sp.exp(x)                    # base case
    return x**n * sp.exp(x) - n * I(n - 1)  # 遞迴關係

print(f"{'n':>3}  {'遞迴公式':>42}  {'SymPy':>42}  一致?")
for n in range(5):
    mine = sp.simplify(sp.expand(I(n)))
    ref  = sp.simplify(sp.integrate(x**n * sp.exp(x), x))
    same = sp.simplify(mine - ref) == 0
    print(f"{n:3d}  {str(sp.factor(mine)):>42}  {str(sp.factor(ref)):>42}  {same}")

# 三角版的遞迴:J_n = -sin^(n-1)x cos x / n + (n-1)/n * J_(n-2)
def J(n):
    if n == 0:
        return x
    if n == 1:
        return -sp.cos(x)
    return -sp.sin(x)**(n-1) * sp.cos(x) / n + sp.Rational(n-1, n) * J(n-2)

print("\n∫ sin^n x dx:")
for n in range(2, 6):
    mine = sp.simplify(J(n))
    ref  = sp.simplify(sp.integrate(sp.sin(x)**n, x))
    print(f"  n={n}  一致? {sp.simplify(mine - ref) == 0}")

In [ ]:
# TODO 學生練習:寫出 ∫ x^n sin(x) dx 的遞迴公式並實作
# 提示:要做兩次分部才會降 1 階(會先冒出 cos)

## Lab 2｜哪些積分「算不出來」

觀念 8 說某些積分沒有初等原函數。SymPy 遇到它們會怎樣?這格讓你親眼看見界線在哪。


In [ ]:
x = sp.Symbol('x')

cases = [
    ("x*exp(x)",      x*sp.exp(x)),
    ("exp(x**2)",     sp.exp(x**2)),
    ("sin(x)/x",      sp.sin(x)/x),
    ("exp(-x**2)",    sp.exp(-x**2)),
    ("x*exp(x**2)",   x*sp.exp(x**2)),
    ("1/log(x)",      1/sp.log(x)),
]
for name, f in cases:
    r = sp.integrate(f, x)
    elementary = not r.has(sp.Integral, sp.erf, sp.Si, sp.li, sp.erfi)
    print(f"  ∫ {name:14s} dx = {str(r)[:46]:46s}  初等? {elementary}")

# 算不出原函數,不代表算不出定積分 —— 數值方法照樣可行
from scipy.integrate import quad
val, err = quad(lambda t: math.exp(-t**2), 0, 1)
print(f"\n∫_0^1 exp(-x^2) dx 數值 = {val:.12f}  (誤差估計 {err:.1e})")
print(f"對照 sympy 的精確值      = {float(sp.integrate(sp.exp(-x**2), (x, 0, 1))):.12f}")